In [273]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error

In [274]:
# Load raw datasets
train_df = pd.read_csv('./data.csv')
test_df = pd.read_csv('./test.csv')

print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

Train shape: (10578, 16), Test shape: (3527, 15)


In [275]:
# Remove extreme rent outliers using percentiles to stabilize the model
Q1 = train_df['rent'].quantile(0.01)
Q3 = train_df['rent'].quantile(0.99)
train_df = train_df[(train_df['rent'] >= Q1) & (train_df['rent'] <= Q3)].copy()

In [276]:
# Combine datasets temporarily to apply cleaning consistently across train and test sets
train_df['is_train'] = 1
test_df['is_train'] = 0
df = pd.concat([train_df, test_df], ignore_index=True)

In [277]:
#checks the baths
df['Baths'].value_counts()

Baths
2       5669
3       3172
1       2073
4       1623
5        966
6        283
7         93
7+        40
none       3
0          1
Name: count, dtype: int64

In [278]:
#see the row with none baths
df['Baths'] = df['Baths'].astype(str).str.strip().replace({'7+': '8', 'none': '1'})
df['Baths'] = df['Baths'].astype(int)
print(df['Baths'].value_counts())

Baths
2    5669
3    3172
1    2076
4    1623
5     966
6     283
7      93
8      40
0       1
Name: count, dtype: int64


In [279]:
# Merge rare property types
if 'Property_type' in df.columns:
    counts = df['Property_type'].value_counts()
    rare_types = counts[counts < 20].index
    df['Property_type'] = df['Property_type'].replace(rare_types, 'Other')

# Merge rare areas
if 'Area' in df.columns:
    counts = df['Area'].value_counts()
    rare_areas = counts[counts < 10].index
    df['Area'] = df['Area'].replace(rare_areas, 'Other')

In [280]:
#change bed column
df['Beds'] = df['Beds'].astype(str).str.lower()
df['Beds'] = df['Beds'].replace({'studio': '1', 'studio+ maid': '1'})
df['Beds'] = df['Beds'].str.extract(r'(\d+)')[0].fillna(1).astype(int)

print(df['Beds'].value_counts())

Beds
2    5474
1    3750
3    2769
4    1415
5     390
6      80
7      40
0       5
Name: count, dtype: int64


In [281]:
#deal with null
df['Availability_date'] = df['Availability_date'].fillna('Unknown')
df['Amenities'] = df['Amenities'].fillna('None')

In [282]:
# Fill text/categorical missing columns
cols_to_fill = ['Title', 'Area', 'Governorate', 'Agent_name', 'Agency', 'Availability_date', 'Amenities']
for col in cols_to_fill:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

In [283]:
df.isnull().sum()

Property_id             0
Offer                   0
URL                     0
Property_type           0
Include_w_e             0
Title                   0
Area                    0
Governorate             0
Beds                    0
Baths                   0
Size                    0
Availability_date       0
Agent_name              0
Agency                  0
Amenities               0
rent                 3527
is_train                0
dtype: int64

In [284]:
# Extract clean datasets back based on 'is_train' flag
train_clean = df[df['is_train'] == 1].copy()
test_clean = df[df['is_train'] == 0].drop(columns=['is_train', 'rent'], errors='ignore').copy()

# Drop rows with missing rent in training data
df_clean = train_clean.dropna(subset=['rent']).copy()

In [285]:
# 10. Separate Cleaned Train and Test Sets properly
train_clean = df[df['is_train'] == 1].copy()
test_clean = df[df['is_train'] == 0].copy()

# Drop 'is_train' and 'rent' and 'Property_id' from features
drop_cols = ['rent', 'Property_id', 'is_train']

X = train_clean.drop(columns=[c for c in drop_cols if c in train_clean.columns]).copy()
y = np.log1p(pd.to_numeric(train_clean['rent'], errors='coerce'))

X_test_final = test_clean.drop(columns=[c for c in drop_cols if c in test_clean.columns]).copy()

In [286]:
# 8. Advanced Feature Engineering (Safe conversion)
if 'Beds' in df.columns and 'Baths' in df.columns and 'Size' in df.columns:
    # Ensure columns are numeric to prevent string division error
    df['Size'] = pd.to_numeric(df['Size'], errors='coerce')
    df['Beds'] = pd.to_numeric(df['Beds'], errors='coerce')
    df['Baths'] = pd.to_numeric(df['Baths'], errors='coerce')
    
    # Total number of rooms
    df['Total_Rooms'] = df['Beds'] + df['Baths']
    
    # Space per room ratio
    df['Space_Per_Room'] = df['Size'] / (df['Total_Rooms'] + 1)
    
    # Interaction between size and number of beds
    df['Size_Beds_Interaction'] = df['Size'] * df['Beds']

In [287]:
# Extract useful keywords from property titles
if 'Title' in df.columns:
    df['Title'] = df['Title'].astype(str).str.lower()
    
    # Check for luxury or premium keywords in the title
    df['Has_Luxury'] = df['Title'].str.contains('lux|luxury|deluxe|premium|vip', case=False, na=False).astype(int)
    
    # Check if the property is furnished
    df['Is_Furnished'] = df['Title'].str.contains('furnished|furnish', case=False, na=False).astype(int)
    
    # Check for parking availability mention in title
    df['Has_Parking'] = df['Title'].str.contains('parking|garage|car park', case=False, na=False).astype(int)

In [288]:
drop_cols = ['rent', 'Property_id']
X = df_clean.drop(columns=[c for c in drop_cols if c in df_clean.columns]).copy()

# Log transformation to stabilize target distribution
y = np.log1p(pd.to_numeric(df_clean['rent'], errors='coerce'))

X_test_final = test_clean.drop(columns=[c for c in drop_cols if c in test_clean.columns]).copy()

In [289]:
for col in X.columns:
    X[col] = X[col].astype(str).astype('category').cat.codes
    if col in X_test_final.columns:
        X_test_final[col] = X_test_final[col].astype(str).astype('category').cat.codes

In [290]:
#  Exclude target ('rent') and unique ID ('Property_id') from features
drop_cols = ['rent', 'Property_id']
X = df_clean.drop(columns=[c for c in drop_cols if c in df_clean.columns]).copy()

In [291]:
# Log-transform the target safely
y = np.log1p(pd.to_numeric(df_clean['rent'], errors='coerce'))

X_test_final = test_clean.drop(columns=[c for c in drop_cols if c in test_clean.columns]).copy()

In [292]:
# Safely encode text/categorical columns into numeric codes
for col in X.columns:
    X[col] = X[col].astype(str).astype('category').cat.codes
    if col in X_test_final.columns:
        X_test_final[col] = X_test_final[col].astype(str).astype('category').cat.codes

In [293]:
# Group-based Imputation for Size based on Area
if 'Area' in df.columns and 'Size' in df.columns:
    df['Size'] = df['Size'].fillna(df.groupby('Area')['Size'].transform('median'))
    
# Fallback for any remaining NaNs in Size using overall median
df['Size'] = df['Size'].fillna(df['Size'].median())

In [294]:
# Impute any missing values with median
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
X_test_imputed = pd.DataFrame(imputer.transform(X_test_final), columns=X_test_final.columns)

In [295]:
# Drop rows where target became NaN from coercion using standard numpy array filtering
valid_idx = ~y.isna().to_numpy()
X_imputed = X_imputed.iloc[valid_idx].reset_index(drop=True)
y = y.iloc[valid_idx].reset_index(drop=True)


In [296]:
# 14. Train-validation split & evaluation with Balanced Random Forest
X_train, X_val, y_train, y_val = train_test_split(X_imputed, y, test_size=0.15, random_state=42)

model = RandomForestRegressor(
    n_estimators=500,
    max_depth=18,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

y_pred_log = model.predict(X_val)
y_pred_original = np.expm1(y_pred_log)
y_val_original = np.expm1(y_val)

print("Validation R2 Score:", r2_score(y_val_original, y_pred_original))
print("Validation MSE:", mean_squared_error(y_val_original, y_pred_original))

Validation R2 Score: 0.7634292217880532
Validation MSE: 29240.78345030517


In [298]:
# 16. Final Prediction & Submission Generation using Random Forest
# Train on the full dataset (X_imputed and y) to maximize learning
model.fit(X_imputed, y)

# Predict on the test set and convert back from log scale
final_preds_log = model.predict(X_test_imputed)
final_preds = np.expm1(final_preds_log)

# Create submission dataframe matching the exact format required
submission = pd.DataFrame({
    'Property_id': test_df['Property_id'], 
    'rent': final_preds
})



In [299]:
# Save to CSV file
submission.to_csv('submission.csv', index=False)
print("Submission file successfully generated and ready for Kaggle!")
print(submission.head())

Submission file successfully generated and ready for Kaggle!
   Property_id         rent
0         4394   418.670700
1         2338   637.393361
2         8531  1026.022324
3         8952   373.663770
4        11064   330.024375
